<< [Search-9-LinearProgramming](Search-9-LinearProgramming.ipynb) | [Index](../README.md) | [App-1-NQueens](../Applications/CSP/App-1-NQueens.ipynb) >>

# Search-10 (C#) : Automates Finis Classiques — jumeau .NET (tranche 1)

**Navigation** : [Index](../README.md) | [Version Python (automates symboliques Z3)](Search-10-SymbolicAutomata.ipynb)

Ce notebook est le **jumeau C# / .NET** de [Search-10-SymbolicAutomata.ipynb](Search-10-SymbolicAutomata.ipynb).
La version Python couvre l'ensemble du sujet (automates finis + automates symboliques avec Z3).
Cette version .NET est livrée en **deux tranches** :

- **Tranche 1 (ce notebook)** — Automates finis classiques : DFA, NFA, opérations ensemblistes.
  Implémentation **manuelle** en C# (cf. §7 du notebook Python : *« Notre approche alternative »* —
  il n'existe pas de librairie .NET équivalente à `automata-lib` qui soit à la fois maintenue et
  didactique ; construire les classes DFA/NFA à la main EST l'objectif pédagogique).
- **Tranche 2 (à suivre)** — Automates symboliques avec `Microsoft.Z3` (NuGet) : prédicats sur
  alphabet infini, intervalles, parité — le cœur « symbolique » du sujet.

## Objectifs d'apprentissage (tranche 1)

À la fin de cette tranche, vous saurez :
1. **Comprendre** la différence entre automate fini déterministe (DFA) et non déterministe (NFA).
2. **Appliquer** — implémenter un DFA et un NFA en C# à partir d'une spécification.
3. **Analyser** — exécuter un automate sur un mot et décider l'acceptation.
4. **Créer** — combiner des automates via les opérations ensemblistes (union, intersection, complément).

### Prérequis

- Bases du C# (classes, génériques, LINQ).
- Notions de théorie des langages (états, alphabet, transitions) — voir §1.

### Durée estimée : 45 minutes


## 1. Introduction aux Automates

### 1.1 Qu'est-ce qu'un automate ?

Un **automate** est un modèle mathématique de calcul. Il se compose de :

| Composant | Description | Notation |
|-----------|-------------|----------|
| **États** | Configurations possibles | $Q = \{q_0, q_1, ...\}$ |
| **Alphabet** | Symboles d'entrée | $\Sigma = \{a, b, ...\}$ |
| **Transitions** | Règles de passage entre états | $\delta : Q \times \Sigma \to Q$ (DFA) ou $\to 2^Q$ (NFA) |
| **État initial** | Point de départ | $q_0 \in Q$ |
| **États finaux** | États d'acceptation | $F \subseteq Q$ |

Un **mot** $w = a_1 a_2 \dots a_n$ est **accepté** si, en partant de $q_0$ et en suivant les
transitions, on termine dans un état final.

### 1.2 DFA vs NFA

- **DFA (Déterministe)** : pour chaque couple (état, symbole), **exactement une** transition.
  $\delta : Q \times \Sigma \to Q$.
- **NFA (Non déterministe)** : pour chaque couple (état, symbole), **zéro, une ou plusieurs**
  transitions. $\delta : Q \times \Sigma \to 2^Q$. Un mot est accepté si **au moins un** chemin
  mène à un état final.

Les deux formalismes ont la même puissance d'expression (tout NFA est convertible en DFA équivalent),
mais le NFA est souvent plus compact pour spécifier un langage.


### 1.3 Exemple introductif — Reconnaissance de "ab"

Soit le langage $L = \{ w \in \{a,b\}^* : w \text{ contient le facteur } ab \}$.
Construisons un NFA qui le reconnaît :

- $q_0$ : état initial (on n'a pas encore vu de `a` candidat).
- $q_1$ : on vient de voir un `a` (candidat au facteur `ab`).
- $q_2$ : état final — on a vu `ab`.

Le non-déterminisme apparaît en $q_0$ : sur `a`, on peut **rester** en $q_0$ (pour chercher un `ab`
plus loin) **ou** passer en $q_1` (tenter le facteur ici). C'est cette liberté qui rend le NFA
plus concis que le DFA équivalent.


In [1]:
// Imports + structures communes : classes DFA et NFA ("hand-rolled").
// Conformement a la section 7 du notebook Python ("Notre approche alternative"),
// on construit les automates a la main : c'est l'objectif pedagogique meme.
using System;
using System.Collections.Generic;
using System.Linq;

// NFA : transitions (etat, symbole) -> ensemble d'etats (non-determinisme).
// Un meme (etat, symbole) peut pointer vers plusieurs etats cibles.
public sealed class NFA {
    public ISet<string> States { get; }
    public ISet<char> Alphabet { get; }
    // transitions[(q, a)] = ensemble d'etats cibles ; absent = ensemble vide.
    public IDictionary<(string, char), ISet<string>> Delta { get; }
    public string InitialState { get; }
    public ISet<string> FinalStates { get; }

    public NFA(ISet<string> states, ISet<char> alphabet,
               IDictionary<(string, char), ISet<string>> delta,
               string initial, ISet<string> finals) {
        States = states; Alphabet = alphabet; Delta = delta;
        InitialState = initial; FinalStates = finals;
    }

    // Etat-atteignable apres lecture d'un mot (union de tous les chemins NFA).
    public ISet<string> EpsilonClosureStep(ISet<string> current, char symbol) {
        var next = new HashSet<string>();
        foreach (var q in current) {
            if (Delta.TryGetValue((q, symbol), out var targets))
                next.UnionWith(targets);
        }
        return next;
    }

    // Accepte w ssi au moins un chemin mene a un etat final.
    public bool Accepts(string word) {
        ISet<string> current = new HashSet<string> { InitialState };
        foreach (var sym in word) {
            current = EpsilonClosureStep(current, sym);
            if (current.Count == 0) return false;
        }
        return current.Overlaps(FinalStates);
    }
}

// DFA : transitions (etat, symbole) -> exactement un etat (determinisme).
public sealed class DFA {
    public ISet<string> States { get; }
    public ISet<char> Alphabet { get; }
    public IDictionary<(string, char), string> Delta { get; }
    public string InitialState { get; }
    public ISet<string> FinalStates { get; }

    public DFA(ISet<string> states, ISet<char> alphabet,
               IDictionary<(string, char), string> delta,
               string initial, ISet<string> finals) {
        States = states; Alphabet = alphabet; Delta = delta;
        InitialState = initial; FinalStates = finals;
    }

    public bool Accepts(string word) {
        var current = InitialState;
        foreach (var sym in word) {
            if (!Delta.TryGetValue((current, sym), out var next))
                return false;  // aucune transition : rejet
            current = next;
        }
        return FinalStates.Contains(current);
    }

    // Complement : L^c = Sigma* \ L. On inverse les etats finaux.
    public DFA Complement() {
        var newFinals = new HashSet<string>(States);
        newFinals.ExceptWith(FinalStates);
        return new DFA(States, Alphabet, Delta, InitialState, newFinals);
    }
}

$@"Bibliotheques definies :
  - DFA  : automate deterministe (Delta : (etat, symbole) -> etat)
  - NFA  : automate non deterministe (Delta : (etat, symbole) -> ensemble d'etats)
Environnement pret.".Display();

The below script needs to be able to find the current output cell; this is an easy method to get it.

Bibliotheques definies :
  - DFA  : automate deterministe (Delta : (etat, symbole) -> etat)
  - NFA  : automate non deterministe (Delta : (etat, symbole) -> ensemble d'etats)
Environnement pret.

## 2. Automates Finis en C#

### 2.2 Création d'un NFA — Reconnaissance de "ab"

Implémentons le NFA de la §1.3. Le non-déterminisme se traduit en C# par un **ensemble de cibles**
pour la transition `('q0', 'a')` = `{'q0', 'q1'}` : sur un `a` en $q_0$, l'automate peut **soit**
rester en $q_0` **soit** passer en $q_1`.


In [2]:
// NFA pour reconnaissance du facteur "ab"
var nfaAb = new NFA(
    states: new HashSet<string> { "q0", "q1", "q2" },
    alphabet: new HashSet<char> { 'a', 'b' },
    delta: new Dictionary<(string, char), ISet<string>> {
        [("q0", 'a')] = new HashSet<string> { "q0", "q1" },  // reste q0 OU va q1
        [("q1", 'b')] = new HashSet<string> { "q2" },         // b apres a -> q2
        [("q2", 'a')] = new HashSet<string> { "q2" },         // puits q2
        [("q2", 'b')] = new HashSet<string> { "q2" },
    },
    initial: "q0",
    finals: new HashSet<string> { "q2" }
);

string[] testWords = { "ab", "aab", "abab", "a", "b", "ba", "aa" };
var sb = new System.Text.StringBuilder();
sb.AppendLine("NFA pour reconnaissance de 'ab'\n");
sb.AppendLine("Alphabet : { a, b }   Etat initial : q0   Etat final : q2\n");
sb.AppendLine($"{"Mot",-8} {"Accepte ?",-12}");
sb.AppendLine(new string('-', 20));
foreach (var w in testWords)
    sb.AppendLine($"{w,-8} {(nfaAb.Accepts(w) ? "OUI" : "non"),-12}");
sb.ToString().Display();

NFA pour reconnaissance de 'ab'

Alphabet : { a, b }   Etat initial : q0   Etat final : q2

Mot      Accepte ?   
--------------------
ab       OUI         
aab      OUI         
abab     OUI         
a        non         
b        non         
ba       non         
aa       non         


**Interprétation : NFA pour reconnaissance de "ab"**

- `ab`, `aab`, `abab` → **acceptés** : ils contiennent le facteur `ab`.
- `a`, `b`, `ba`, `aa` → **rejetés** : aucun ne contient `ab`.

Le non-déterminisme (deux cibles pour `('q0','a')`) est résolu par l'exploration simultanée de
**tous les chemins** (`EpsilonClosureStep` cumule les cibles). Le mot est accepté dès qu'**un** chemin
se termine en $q_2$.


### 2.3 Création d'un DFA — Mots avec un nombre pair de 'a'

Le langage $L = \{ w \in \{a,b\}^* : w \text{ contient un nombre pair de } a \}$ se code
naturellement par un DFA à **deux états** : $q_{even}$ (pair, final) et $q_{odd}$ (impair).
Sur `a` on change de parité, sur `b` on ne change pas.


In [3]:
// DFA pour nombre pair de 'a'
var dfaEvenA = new DFA(
    states: new HashSet<string> { "q_even", "q_odd" },
    alphabet: new HashSet<char> { 'a', 'b' },
    delta: new Dictionary<(string, char), string> {
        [("q_even", 'a')] = "q_odd",   // a change la parite
        [("q_even", 'b')] = "q_even",  // b non
        [("q_odd", 'a')]  = "q_even",
        [("q_odd", 'b')]  = "q_odd",
    },
    initial: "q_even",
    finals: new HashSet<string> { "q_even" }  // pair = accepte
);

string[] testWordsDfa = { "", "a", "aa", "aaa", "b", "ab", "aba", "bab" };
var sb2 = new System.Text.StringBuilder();
sb2.AppendLine("DFA pour nombre pair de 'a'\n");
sb2.AppendLine($"{"Mot",-8} {"#a",-4} {"Accepte ?",-12}");
sb2.AppendLine(new string('-', 26));
foreach (var w in testWordsDfa) {
    int na = w.Count(c => c == 'a');
    sb2.AppendLine($"{(w == "" ? "(vide)" : w),-8} {na,-4} {(dfaEvenA.Accepts(w) ? "OUI" : "non"),-12}");
}
sb2.ToString().Display();

DFA pour nombre pair de 'a'

Mot      #a   Accepte ?   
--------------------------
(vide)   0    OUI         
a        1    non         
aa       2    OUI         
aaa      3    non         
b        0    OUI         
ab       1    non         
aba      2    OUI         
bab      1    non         


**Interprétation : DFA nombre pair de 'a'**

Le mot vide `""` est accepté (zéro `a`, zéro est pair). `aaa` (3 `a`) est rejeté, `aa` (2) accepté.
Ce DFA est **minimal** : deux états suffisent car seule la parité du compte de `a` importe.


### Exercice : NFA pour mots se terminant par "ba"

Construisez un NFA reconnaissant le langage $L = \{ w \in \{a,b\}^* : w \text{ se termine par } ba \}$.

**Indices** :
- *Étape 1* : trois états suffisent — un état "en cours" $q_0$, un état "on vient de voir `b`" $q_1$,
  et un état final $q_2$ ("on vient de voir `ba`").
- *Étape 2* : attention — sur `b` en $q_1$, où retourner ? (le nouveau `b` peut être le début du
  suffixe `ba` final).
- *Étape 3* : testez avec `ba`, `aba`, `bba` (acceptés) et `ab`, `aa`, `bb` (rejetés).


In [4]:
// Exercice : NFA pour mots se terminant par "ba"
// TODO etudiant : construire le NFA et le stocker dans 'nfaBa'.
// Etape 1 : definir etats / alphabet / transitions / initial / finals
// Etape 2 : caser le non-determinisme sur 'b' (le b peut etre debut de suffixe OU non)
// Etape 3 : tester ba / aba / bba (acceptes) vs ab / aa / bb (rejetes)
NFA nfaBa = null;  // TODO etudiant : remplacer par la solution NFA
Console.WriteLine("Exercice a completer : NFA pour mots se terminant par 'ba'");

Exercice a completer : NFA pour mots se terminant par 'ba'


### 2.4 Opérations sur les automates

Les langages reconnus par des automates sont **clos** par les opérations ensemblistes :

| Opération | Langage | Construction sur DFA |
|-----------|---------|----------------------|
| **Union** $L_1 \cup L_2$ | mots acceptés par $L_1$ **ou** $L_2$ | produit cartésien des états |
| **Intersection** $L_1 \cap L_2$ | mots acceptés par $L_1$ **et** $L_2$ | produit cartésien, final = couple final×final |
| **Complément** $\overline{L}$ | mots **non** acceptés | inverser les états finaux (DFA complet) |

Le **complément** est immédiat sur un DFA **complet** (toutes les transitions définies) : on
inverse simplement l'ensemble des états finaux (méthode `DFA.Complement()` ci-dessus). L'union et
l'intersection passent par le **produit cartésien** de deux automates.


In [5]:
// Demonstration : complement du DFA "pair de a".
// L_complement = mots a nombre IMPAIR de 'a'.
var dfaOddA = dfaEvenA.Complement();

// Intersection par produit cartesien : (p,q) final ssi p final pour L1 ET q final pour L2.
// Illustration : L1 = "contient ab" (NFA -> on prend l'acceptation NFA directe)
//                L2 = "pair de a"    (DFA)
// On montre ici le complement + un produit cartesien simple : pair de a ET se termine par 'b'.
// (DFA_pair_a x DFA_termine_par_b : deux DFA -> produit direct)
var dfaEndsB = new DFA(
    states: new HashSet<string> { "p0", "p1" },  // p0 = ne termine pas par b (ou vide), p1 = termine par b
    alphabet: new HashSet<char> { 'a', 'b' },
    delta: new Dictionary<(string, char), string> {
        [("p0", 'a')] = "p0",
        [("p0", 'b')] = "p1",
        [("p1", 'a')] = "p0",
        [("p1", 'b')] = "p1",
    },
    initial: "p0",
    finals: new HashSet<string> { "p1" }
);

// Produit cartesien DFA_pair_a x DFA_ends_b : (etat_pair, etat_ends)
var prodStates = new HashSet<string>();
var prodDelta = new Dictionary<(string, char), string>();
foreach (var s1 in dfaEvenA.States)
    foreach (var s2 in dfaEndsB.States) {
        prodStates.Add($"{s1}|{s2}");
        foreach (var sym in dfaEvenA.Alphabet)
            prodDelta[($"{s1}|{s2}", sym)] = $"{dfaEvenA.Delta[(s1, sym)]}|{dfaEndsB.Delta[(s2, sym)]}";
    }
// Final ssi les deux composantes sont finales.
var prodFinals = new HashSet<string>(
    from s1 in dfaEvenA.FinalStates
    from s2 in dfaEndsB.FinalStates
    select $"{s1}|{s2}"
);
var dfaInter = new DFA(prodStates, dfaEvenA.Alphabet, prodDelta, $"q_even|p0", prodFinals);

string[] testWordsOp = { "b", "ab", "bb", "aab", "bab", "aa", "aabab" };
var sb3 = new System.Text.StringBuilder();
sb3.AppendLine("Operations sur les automates\n");
sb3.AppendLine("L1 = 'nombre pair de a'   L2 = 'se termine par b'   L1 n L2\n");
sb3.AppendLine($"{"Mot",-8} {"L1",-5} {"L2",-5} {"L1nL2",-7}");
sb3.AppendLine(new string('-', 28));
foreach (var w in testWordsOp) {
    bool l1 = dfaEvenA.Accepts(w), l2 = dfaEndsB.Accepts(w), lint = dfaInter.Accepts(w);
    sb3.AppendLine($"{(w == "" ? "(vide)" : w),-8} {(l1 ? "OUI" : "non"),-5} {(l2 ? "OUI" : "non"),-5} {(lint ? "OUI" : "non"),-7}");
}
sb3.AppendLine("\nVerification coherence : L1 n L2 == L1 logique AND L2 sur chaque mot.");
bool coherent = testWordsOp.All(w => dfaInter.Accepts(w) == (dfaEvenA.Accepts(w) && dfaEndsB.Accepts(w)));
sb3.AppendLine($"Coherence produit cartesien : {(coherent ? "OK (intersection = AND logique)" : "ECHEC")}");
sb3.ToString().Display();

Operations sur les automates

L1 = 'nombre pair de a'   L2 = 'se termine par b'   L1 n L2

Mot      L1    L2    L1nL2  
----------------------------
b        OUI   OUI   OUI    
ab       non   OUI   non    
bb       OUI   OUI   OUI    
aab      OUI   OUI   OUI    
bab      non   OUI   non    
aa       OUI   non   non    
aabab    non   OUI   non    

Verification coherence : L1 n L2 == L1 logique AND L2 sur chaque mot.
Coherence produit cartesien : OK (intersection = AND logique)


**Interprétation : opérations sur les automates**

- L'**intersection** par produit cartésien donne exactement le ET logique mot à mot (vérifié :
  `Coherence OK`). C'est la construction standard de fermeture des langages réguliers.
- Le **complément** (`dfaEvenA.Complement()` → nombres impairs de `a`) ne fonctionne que parce que le
  DFA est **complet** (toute transition définie). Sur un NFA, il faudrait d'abord déterminiser.
- Ces opérations sont la base de la **vérification de propriétés** (§5 du notebook Python) : on
  construit l'automate du système, celui de la propriété, et on teste l'intersection / le complément.

### 2.5 Limite des automates finis classiques

Un automate fini ne peut **pas** compter arbitrairement : il ne peut reconnaître ni
$\{a^n b^n : n \geq 0\}$ (comptage imbriqué), ni $\{w : w \text{ est un carré}\}$.
Plus précisément, un DFA à $k$ états ne peut pas distinguer deux mots qui passent par le **même**
état après un préfixe assez long (pumping lemma). C'est cette limite qui motive les
**automates symboliques** : au lieu d'énumérer des symboles concrets, on manipule des **prédicats**
sur un alphabet potentiellement infini (entiers, par exemple). Ce sera l'objet de la **tranche 2**
(avec `Microsoft.Z3`).


---

## Bilan de la tranche 1

### Concepts clés
- **DFA** (déterministe) : une transition par (état, symbole) — implémenté comme
  `Dictionary<(string,char), string>`.
- **NFA** (non déterministe) : plusieurs transitions possibles — implémenté comme
  `Dictionary<(string,char), HashSet<string>>` ; l'acceptation explore **tous** les chemins.
- **Opérations ensemblistes** : complément (inversion des finaux sur DFA complet), union et
  intersection (produit cartésien). Les langages réguliers y sont **clos**.
- **Pumping lemma** : borne fondamentale — un automate fini ne peut pas « compter » au-delà de son
  nombre d'états.

### Ce que la tranche 2 ajoutera
- Automates **symboliques** : transitions étiquetées par des **prédicats Z3** sur un alphabet infini.
- Exemples : intervalle $[10, 100]$, nombres pairs, multiples de 5 dans un intervalle.
- Application : **vérification de propriétés** (système de porte, invariant d'état, mini-Sudoku).

### Version de référence
Le notebook Python complet ([Search-10-SymbolicAutomata.ipynb](Search-10-SymbolicAutomata.ipynb))
couvre les deux tranches avec la librairie `automata-lib` (DFA/NFA) et `z3-solver` (symbolique).

> **Honnêtement** : l'implémentation manuelle des classes DFA/NFA en C# n'est **pas** une lacune —
> c'est un choix pédagogique délibéré (cf. §7 du notebook Python). Il n'existe pas de librairie .NET
> maintenue, didactique et équivalente à `automata-lib` ; construire les automates à la main force à
> expliciter chaque concept (états, transitions, acceptation). La tranche 2 utilisera bien la vraie
> librairie SOTA `Microsoft.Z3` pour la partie symbolique.
